# 最近可达点与投影

目标双声道采样值为 (3,1)，只能输出 (c,c)。比较所有 c 的误差，而不是只试几份。

依赖：NumPy、Matplotlib。按顺序运行，先在纸上预测再核对；图内使用英文标签。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def check_prediction(value, expected, label):
    if value is None:
        print(label + '：尚未填写，请先在纸上计算，再替换 None。')
        return None
    try:
        proposed = np.asarray(value, dtype=float)
    except (TypeError, ValueError, OverflowError):
        print(label + '：请填写数值、数值列表或数组。')
        return False
    reference = np.asarray(expected, dtype=float)
    if proposed.shape != reference.shape:
        print(label + '：尺寸应为 ' + str(reference.shape) + '，请检查行列。')
        return False
    if not np.all(np.isfinite(proposed)):
        print(label + '：请检查是否除以零或产生了非有限数。')
        return False
    correct = bool(np.allclose(proposed, reference))
    print(label + ('：核对正确。' if correct else '：尚不一致，请回到方程或内积检查。'))
    return correct


## 先配方，再看曲线

先把误差平方写成 2(c-2)^2+2。图中的最小点和几何中的垂足是否对应？


In [ ]:
u = np.array([1.,1.])
b = np.array([3.,1.])
coefficient = (u @ b)/(u @ u)
p = coefficient*u
r = b-p
assert np.allclose(u @ r,0)
c = np.linspace(-1,4,301)
errors = ((b[:,None]-u[:,None]*c)**2).sum(axis=0)
fig,axes = plt.subplots(1,2,figsize=(10,4.5),dpi=150)
axes[0].plot(c,errors,color='#2878b5')
axes[0].scatter(coefficient,r @ r,color='#c53351',zorder=4)
axes[0].set(xlabel='common sample c',ylabel='squared error',title='Best common signal')
axes[1].plot([-1,4],[-1,4],color='#2878b5',label='reachable (c,c)')
axes[1].scatter(*b,color='#c53351',label='target (3,1)')
axes[1].scatter(*p,color='#359969',label='projection (2,2)')
axes[1].plot([p[0],b[0]],[p[1],b[1]],'--',color='#d97922',label='orthogonal residual')
axes[1].set(xlim=(-1,4),ylim=(-1,4),aspect='equal',xlabel='L',ylabel='R',title='Closest reachable point')
axes[1].legend()
for ax in axes: ax.grid(alpha=.2)
fig.tight_layout()
plt.show()


## 直接改变候选 c

预测误差，然后修改 candidate。


In [ ]:
candidate = 1.5
error = b-candidate*u
print('候选输出:',candidate*u,'误差平方:',error @ error)
assert error @ error >= r @ r - 1e-12


## 投影不一致的报表

三个字段同等对待时，为什么不只修改总额？


In [ ]:
report = np.array([120.,80.,210.])
normal = np.array([-1.,-1.,1.])
residual = normal*(normal @ report)/(normal @ normal)
consistent = report-residual
print('最近一致报表:',consistent,'残差:',residual)
assert np.allclose(consistent[2],consistent[0]+consistent[1])
C = np.array([[1.,0.],[0.,1.],[1.,1.]])
assert np.allclose(C.T @ residual,0)


## 投到平面，不只是只修第三项

目标 (2,3,6)，可达平面 z=x+y。单位基为 q1=(1,0,1)/√2、q2=(-1,2,1)/√6。

先求两个坐标，再合回投影。把最近点按三个坐标填入 `predicted_projection`。


In [ ]:
Qplane = np.column_stack((np.array([1.,0.,1.])/np.sqrt(2),
                         np.array([-1.,2.,1.])/np.sqrt(6)))
target_plane = np.array([2.,3.,6.])
predicted_projection = None
check_prediction(predicted_projection, [7/3,10/3,17/3], '平面投影预测')
assert np.allclose(Qplane.T @ Qplane, np.eye(2))
coordinates_plane = Qplane.T @ target_plane
projected_plane = Qplane @ coordinates_plane
residual_plane = target_plane - projected_plane
print('coordinates:', coordinates_plane, 'projection:', projected_plane)
print('residual:', residual_plane)
assert np.allclose(projected_plane, [7/3,10/3,17/3])
assert np.allclose(Qplane.T @ residual_plane, 0)
assert np.isclose(projected_plane[2], projected_plane[0]+projected_plane[1])
Pplane = Qplane @ Qplane.T
assert np.allclose(Pplane @ Pplane, Pplane)
assert np.allclose(Pplane.T, Pplane)
assert not np.allclose(Pplane, np.eye(3))


## 比较候选，并核对勾股拆分

候选 (2,3,5) 只改总额。先预测它的误差平方与最近点的误差平方。再找另一个平面内候选：不必猜它最优，但必须解释为什么误差不会更小。


In [ ]:
for alternative in [np.array([2.,3.,5.]), np.array([1.,4.,5.])]:
    error2 = np.linalg.norm(target_plane-alternative)**2
    best2 = np.linalg.norm(residual_plane)**2
    extra2 = np.linalg.norm(projected_plane-alternative)**2
    print('candidate:', alternative, 'error:', error2, '=', best2, '+', extra2)
    assert np.isclose(error2, best2+extra2)
    assert error2 >= best2-1e-12
assert np.isclose(np.linalg.norm(residual_plane)**2, 1/3)


<details><summary>核对</summary>
最近点为 (7/3,10/3,17/3)，残差为 (-1,-1,1)/3，平方误差 1/3。只修第三项误差为 1。正交残差加上任意平面内的额外变化，平方长度相加，所以不可能更好。Q是三行两列，QQ^T是投影，不是单位矩阵。
</details>


## 合上实验，自己解释

先配方求单声道的最优系数，再从两个单位基的内积求平面投影。用勾股分解证明任意另一候选都不更好；说明QQ^T为什么不是三维单位矩阵。若只允许改报表总额，原来的同权投影为何不再是指定问题的答案？
